In [ ]:
# Occupancy Test for 4500 images taken from video frames. The test results are saved in each model named folder in occupancy_results_YOLOvxx.
# In final cell the occupancy obtained for each percentage and their average confidence score obtained in displayed and result is stored in occupancy_summary.xlsx

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
TRAIN_LABELS = "/content/drive/MyDrive/Major_Project/Test_4500/images"
VAL_LABELS  = "/content/drive/MyDrive/Major_Project/Test_4500/labels"

!ls "$TRAIN_LABELS" | wc -l
!ls "$VAL_LABELS" | wc -l

4500
4500


In [3]:
# ============================================================
# Road Hazard Occupancy Detection — YOLOv8s
# Metrics: Occupancy % + Confidence Score
# GT Check: Ground_Truth | Predicted_Class | Correct
# Labels  : YOLO .txt files in /labels folder (class_id x y w h)
# ============================================================

import os
import re
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────

CLASS_NAMES = {
    0: "Speed_Breaker",
    1: "Slippery_Road",
    2: "School_Zone"
}

IMAGE_FOLDER     = "/content/drive/MyDrive/Major_Project/Test_4500/images"
LABELS_FOLDER    = "/content/drive/MyDrive/Major_Project/Test_4500/labels"
MODEL_PATH       = "/content/drive/MyDrive/Major_Project/3Classes/MobileVit_SEBlock/mobilevit_SEblock_best_float32.tflite"
RESULTS_FOLDER   = "/content/drive/MyDrive/Major_Project/Occupancy_Results/Yolo_MobileVIT"
ANNOTATED_FOLDER = os.path.join(RESULTS_FOLDER, "annotated_images")

os.makedirs(RESULTS_FOLDER, exist_ok=True)
os.makedirs(ANNOTATED_FOLDER, exist_ok=True)

CONFIDENCE_THRESHOLD = 0.25
NMS_IOU_THRESHOLD    = 0.45


# ─────────────────────────────────────────────
# GROUND TRUTH READER
# ─────────────────────────────────────────────
# Reads YOLO .txt label file for a given image.
# Each line format: class_id cx cy w h
# We collect ALL annotated class_ids (one image can
# have multiple signs), and use them to verify prediction.
# ─────────────────────────────────────────────

def get_ground_truth(img_name):
    """
    Returns:
        gt_classes  → list of class names annotated in label file
        gt_str      → comma-separated string for CSV display
    """
    base       = os.path.splitext(img_name)[0]
    label_path = os.path.join(LABELS_FOLDER, base + ".txt")

    if not os.path.exists(label_path):
        return [], "No Label File"

    gt_classes = []
    with open(label_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            cls_id = int(line.split()[0])
            name   = CLASS_NAMES.get(cls_id, f"Unknown_{cls_id}")
            if name not in gt_classes:
                gt_classes.append(name)

    if not gt_classes:
        return [], "Empty Label"

    return gt_classes, ", ".join(gt_classes)


# ─────────────────────────────────────────────
# OCCUPANCY + HELPERS
# ─────────────────────────────────────────────

def compute_occupancy(box, img_w, img_h):
    """
    occupancy % = (box_w × box_h) / (img_w × img_h) × 100
    background % = 100 - occupancy %
    Both always sum to 100.
    """
    bw  = max(0, box[2] - box[0])
    bh  = max(0, box[3] - box[1])
    occ = round((bw * bh) / (img_w * img_h) * 100, 2)
    return occ, round(100.0 - occ, 2)


def occupancy_level(pct):
    if pct >= 50: return "VERY CLOSE"
    if pct >= 30: return "CLOSE"
    if pct >= 15: return "MODERATE"
    if pct >= 5:  return "FAR"
    return "VERY FAR"


def xywh_to_xyxy(cx, cy, w, h, img_w, img_h):
    x1 = int((cx - w / 2) * img_w)
    y1 = int((cy - h / 2) * img_h)
    x2 = int((cx + w / 2) * img_w)
    y2 = int((cy + h / 2) * img_h)
    return (max(0, x1), max(0, y1),
            min(img_w - 1, x2), min(img_h - 1, y2))


def apply_nms(boxes_xyxy, scores):
    if not boxes_xyxy:
        return []
    boxes_xywh = [[x1, y1, x2 - x1, y2 - y1] for x1, y1, x2, y2 in boxes_xyxy]
    indices    = cv2.dnn.NMSBoxes(boxes_xywh, scores,
                                   CONFIDENCE_THRESHOLD, NMS_IOU_THRESHOLD)
    return indices.flatten().tolist() if len(indices) else []


# ─────────────────────────────────────────────
# ANNOTATION DRAWING
# ─────────────────────────────────────────────

def draw_annotations(image, detections, img_name, gt_str, correct_str):
    vis  = image.copy()
    h, w = vis.shape[:2]

    COLORS = {
        0: (0, 165, 255),   # Speed Breaker → orange
        1: (255, 0, 200),   # Slippery Road → pink
        2: (255, 220, 0),   # School Zone   → cyan
    }

    for det in detections:
        x1, y1, x2, y2 = det["box"]
        color = COLORS.get(det["class_id"], (200, 200, 200))

        cv2.rectangle(vis, (x1, y1), (x2, y2), color, 2)

        line1 = f"{CLASS_NAMES[det['class_id']]}"
        line2 = f"Conf: {det['confidence']:.4f}"
        line3 = f"Occ : {det['occupancy_pct']:.2f}%  [{det['level']}]"

        ty = max(y1 - 50, 14)
        for i, line in enumerate([line1, line2, line3]):
            cv2.putText(vis, line, (x1, ty + i * 16),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 2)

        overlay = vis.copy()
        cv2.rectangle(overlay, (x1, y1), (x2, y2), color, -1)
        cv2.addWeighted(overlay, 0.15, vis, 0.85, 0, vis)

    # Top banner — colour reflects correctness
    if correct_str == "Correct":
        bg_color = (0, 140, 0)
    elif correct_str == "Wrong":
        bg_color = (0, 0, 180)
    else:
        bg_color = (60, 60, 60)

    if detections:
        best   = max(detections, key=lambda d: d["confidence"])
        banner = (f"GT:{gt_str}  |  Pred:{CLASS_NAMES[best['class_id']]}  "
                  f"Conf={best['confidence']:.4f}  "
                  f"Occ={best['occupancy_pct']:.2f}%  "
                  f"[{correct_str}]")
    else:
        banner = f"GT:{gt_str}  |  No Detection  [{correct_str}]"

    cv2.rectangle(vis, (0, 0), (w, 36), bg_color, -1)
    cv2.putText(vis, banner, (6, 24),
                cv2.FONT_HERSHEY_SIMPLEX, 0.48, (255, 255, 255), 2)

    # Bottom occupancy bar
    if detections:
        best  = max(detections, key=lambda d: d["confidence"])
        bar_y = h - 22
        occ_w = int(w * best["occupancy_pct"] / 100)
        cv2.rectangle(vis, (0, bar_y), (w, bar_y + 18), (50, 50, 50), -1)
        cv2.rectangle(vis, (0, bar_y), (occ_w, bar_y + 18), (0, 80, 220), -1)
        cv2.putText(vis,
                    f"Occupancy {best['occupancy_pct']:.2f}% | Background {best['background_pct']:.2f}%",
                    (6, bar_y + 13),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1)

    return vis


# ─────────────────────────────────────────────
# LOAD MODEL
# ─────────────────────────────────────────────

interpreter = tf.lite.Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()
input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()
input_h = input_details[0]['shape'][1]
input_w = input_details[0]['shape'][2]

print(f"Model input         : {input_w} × {input_h}")
print(f"Classes             : {list(CLASS_NAMES.values())}")
print(f"Confidence threshold: {CONFIDENCE_THRESHOLD}")
print(f"Labels folder       : {LABELS_FOLDER}")
print(f"Occupancy formula   : (box_w × box_h) / (img_w × img_h) × 100")
print(f"Note                : Occupancy % + Background % = 100% always")
print("=" * 110)
print(f"{'Image':<32} {'Ground_Truth':<18} {'Predicted_Class':<18} "
      f"{'Correct':<10} {'Confidence':>11} {'Occupancy%':>11} {'Background%':>12}")
print("=" * 110)


# ─────────────────────────────────────────────
# MAIN INFERENCE LOOP
# ─────────────────────────────────────────────

data = []

for img_name in sorted(os.listdir(IMAGE_FOLDER)):
    if not img_name.lower().endswith((".jpg", ".png", ".jpeg")):
        continue

    image = cv2.imread(os.path.join(IMAGE_FOLDER, img_name))
    if image is None:
        print(f"⚠  Could not read {img_name}")
        continue

    img_h, img_w = image.shape[:2]

    # ── Ground truth from YOLO label file ────────────────────
    gt_classes, gt_str = get_ground_truth(img_name)

    # ── Preprocess ────────────────────────────────────────────
    inp = cv2.resize(image, (input_w, input_h))
    inp = cv2.cvtColor(inp, cv2.COLOR_BGR2RGB)
    inp = np.expand_dims(inp, 0).astype(np.float32) / 255.0

    interpreter.set_tensor(input_details[0]['index'], inp)
    interpreter.invoke()

    # ── Parse output ──────────────────────────────────────────
    raw = interpreter.get_tensor(output_details[0]['index'])
    raw = raw.transpose(0, 2, 1)[0]       # → (8400, 7)

    raw_boxes  = raw[:, :4]
    raw_scores = raw[:, 4:7]

    if raw_scores.min() < 0 or raw_scores.max() > 1:
        raw_scores = 1.0 / (1.0 + np.exp(-raw_scores))

    class_ids   = np.argmax(raw_scores, axis=1)
    confidences = np.max(raw_scores, axis=1)

    # ── Filter + NMS ──────────────────────────────────────────
    mask    = confidences >= CONFIDENCE_THRESHOLD
    f_boxes = raw_boxes[mask]
    f_conf  = confidences[mask].tolist()
    f_cls   = class_ids[mask].tolist()

    boxes_px = [xywh_to_xyxy(b[0], b[1], b[2], b[3], img_w, img_h)
                for b in f_boxes]
    keep     = apply_nms(boxes_px, f_conf)

    # ── Build detections list ─────────────────────────────────
    detections = []
    for i in keep:
        box    = boxes_px[i]
        conf   = round(float(f_conf[i]), 4)
        cls_id = int(f_cls[i])
        occ_pct, bg_pct = compute_occupancy(box, img_w, img_h)
        detections.append({
            "box":            box,
            "class_id":       cls_id,
            "confidence":     conf,
            "occupancy_pct":  occ_pct,
            "background_pct": bg_pct,
            "level":          occupancy_level(occ_pct),
        })

    detections = sorted(detections, key=lambda d: d["confidence"], reverse=True)
    best       = detections[0] if detections else None

    # ── Predicted class ───────────────────────────────────────
    if best:
        pred_class = CLASS_NAMES[best["class_id"]]
        pred_conf  = best["confidence"]
        occ_pct    = best["occupancy_pct"]
        bg_pct     = best["background_pct"]
        level      = best["level"]
    else:
        pred_class = "No Detection"
        pred_conf  = 0.0
        occ_pct    = 0.0
        bg_pct     = 100.0
        level      = "NO DETECTION"

    # ── Correct / Wrong ───────────────────────────────────────
    # Correct = predicted class is in the list of GT classes
    if gt_str in ("No Label File", "Empty Label"):
        correct_str = "No GT"
    elif pred_class == "No Detection":
        correct_str = "Wrong"        # missed detection = error
    elif pred_class in gt_classes:
        correct_str = "Correct"
    else:
        correct_str = "Wrong"

    # ── Print row ─────────────────────────────────────────────
    icon = {"Correct": "✅", "Wrong": "❌"}.get(correct_str, "❓")
    print(f"{icon} {img_name:<32} {gt_str:<18} {pred_class:<18} "
          f"{correct_str:<10} {pred_conf:>11.4f} "
          f"{occ_pct:>10.2f}% {bg_pct:>11.2f}%")

    # ── Distance level ────────────────────────────────────────
    dist_match = re.search(r'_l(\d+)_', img_name)
    dist_level = dist_match.group(1) if dist_match else "unknown"

    # ── Record one row per detection (or one "no detection" row)
    if detections:
        for det in detections:
            data.append({
                "Image":            img_name,
                "Ground_Truth":     gt_str,
                "Predicted_Class":  CLASS_NAMES[det["class_id"]],
                "Correct":          correct_str if det == detections[0] else "—",
                "Confidence":       det["confidence"],
                "Occupancy_%":      det["occupancy_pct"],
                "Background_%":     det["background_pct"],
                "Occ+BG_Check":     round(det["occupancy_pct"] + det["background_pct"], 2),
                "Proximity_Level":  det["level"],
                "Box_x1":           det["box"][0],
                "Box_y1":           det["box"][1],
                "Box_x2":           det["box"][2],
                "Box_y2":           det["box"][3],
                "Image_W":          img_w,
                "Image_H":          img_h,
                "Distance_Level":   dist_level,
            })
    else:
        data.append({
            "Image":            img_name,
            "Ground_Truth":     gt_str,
            "Predicted_Class":  "No Detection",
            "Correct":          correct_str,
            "Confidence":       0.0,
            "Occupancy_%":      0.0,
            "Background_%":     100.0,
            "Occ+BG_Check":     100.0,
            "Proximity_Level":  "NO DETECTION",
            "Box_x1": 0, "Box_y1": 0, "Box_x2": 0, "Box_y2": 0,
            "Image_W": img_w, "Image_H": img_h,
            "Distance_Level":   dist_level,
        })

    # ── Save annotated image ──────────────────────────────────
    vis = draw_annotations(image, detections, img_name, gt_str, correct_str)
    cv2.imwrite(os.path.join(ANNOTATED_FOLDER, img_name), vis)

print("=" * 110)

# ─────────────────────────────────────────────
# BUILD DATAFRAME
# ─────────────────────────────────────────────

df = pd.DataFrame(data)

# For summary stats, use only the primary detection row per image
# (rows where Correct is not "—")
primary = df[df["Correct"] != "—"].copy()

total       = primary["Image"].nunique()
evaluatable = primary[~primary["Correct"].isin(["No GT"])]
correct_n   = len(evaluatable[evaluatable["Correct"] == "Correct"])
wrong_n     = len(evaluatable[evaluatable["Correct"] == "Wrong"])
no_gt_n     = len(primary[primary["Correct"] == "No GT"])
accuracy    = (correct_n / len(evaluatable) * 100) if len(evaluatable) else 0

# ─────────────────────────────────────────────
# TALLY ROWS — appended at bottom of CSV
# ─────────────────────────────────────────────

separator = {col: "─" * 10 for col in df.columns}
separator["Image"] = "─" * 28

tally_rows = [
    separator,
    {"Image": "TOTAL IMAGES",          "Ground_Truth": str(total),      "Predicted_Class": "", "Correct": "", "Confidence": "", "Occupancy_%": "", "Background_%": "", "Occ+BG_Check": "", "Proximity_Level": "", "Box_x1": "", "Box_y1": "", "Box_x2": "", "Box_y2": "", "Image_W": "", "Image_H": "", "Distance_Level": ""},
    {"Image": "TOTAL CORRECT ✅",      "Ground_Truth": str(correct_n),  "Predicted_Class": "", "Correct": "", "Confidence": "", "Occupancy_%": "", "Background_%": "", "Occ+BG_Check": "", "Proximity_Level": "", "Box_x1": "", "Box_y1": "", "Box_x2": "", "Box_y2": "", "Image_W": "", "Image_H": "", "Distance_Level": ""},
    {"Image": "TOTAL WRONG ❌",        "Ground_Truth": str(wrong_n),    "Predicted_Class": "", "Correct": "", "Confidence": "", "Occupancy_%": "", "Background_%": "", "Occ+BG_Check": "", "Proximity_Level": "", "Box_x1": "", "Box_y1": "", "Box_x2": "", "Box_y2": "", "Image_W": "", "Image_H": "", "Distance_Level": ""},
    {"Image": "NO GROUND TRUTH ❓",    "Ground_Truth": str(no_gt_n),    "Predicted_Class": "", "Correct": "", "Confidence": "", "Occupancy_%": "", "Background_%": "", "Occ+BG_Check": "", "Proximity_Level": "", "Box_x1": "", "Box_y1": "", "Box_x2": "", "Box_y2": "", "Image_W": "", "Image_H": "", "Distance_Level": ""},
    {"Image": "ACCURACY",              "Ground_Truth": f"{accuracy:.2f}%", "Predicted_Class": "", "Correct": "", "Confidence": "", "Occupancy_%": "", "Background_%": "", "Occ+BG_Check": "", "Proximity_Level": "", "Box_x1": "", "Box_y1": "", "Box_x2": "", "Box_y2": "", "Image_W": "", "Image_H": "", "Distance_Level": ""},
]

df_final = pd.concat([df, pd.DataFrame(tally_rows)], ignore_index=True)
csv_path = os.path.join(RESULTS_FOLDER, "occupancy_results.csv")
df_final.to_csv(csv_path, index=False)

# ─────────────────────────────────────────────
# PRINT SUMMARY
# ─────────────────────────────────────────────

detected = primary[primary["Predicted_Class"] != "No Detection"]

print(f"\n✅  Done! Processed {total} images")
print(f"📂  CSV       : {csv_path}")
print(f"🖼   Annotated : {ANNOTATED_FOLDER}\n")

print("=" * 60)
print("✅❌  GROUND TRUTH vs PREDICTED — SUMMARY")
print("=" * 60)
print(f"  Total Images          : {total}")
print(f"  Evaluatable (GT known): {len(evaluatable)}")
print(f"  ✅  Correct           : {correct_n}")
print(f"  ❌  Wrong / Missed    : {wrong_n}")
print(f"  ❓  No Ground Truth   : {no_gt_n}")
print(f"  🎯  Accuracy          : {accuracy:.2f}%")

print("\n" + "=" * 60)
print("📊  CONFIDENCE SCORE SUMMARY")
print("=" * 60)
if len(detected):
    print(f"  Mean   : {detected['Confidence'].mean():.4f}")
    print(f"  Median : {detected['Confidence'].median():.4f}")
    print(f"  Min    : {detected['Confidence'].min():.4f}")
    print(f"  Max    : {detected['Confidence'].max():.4f}")

print("\n" + "=" * 60)
print("📐  OCCUPANCY % SUMMARY")
print("=" * 60)
print("  Note: Occupancy % + Background % = 100% for every detection\n")
if len(detected):
    print(f"  Mean Occupancy   : {detected['Occupancy_%'].mean():.2f}%")
    print(f"  Mean Background  : {detected['Background_%'].mean():.2f}%")
    print(f"  Min Occupancy    : {detected['Occupancy_%'].min():.2f}%")
    print(f"  Max Occupancy    : {detected['Occupancy_%'].max():.2f}%")
    check_ok = (df[df["Occ+BG_Check"] != ""]["Occ+BG_Check"].astype(float) == 100.0).all()
    print(f"\n  ✅ Occ% + BG% = 100% check : {'PASSED' if check_ok else 'FAILED'}")

print("\n" + "=" * 60)
print("🏷   CLASS-WISE ACCURACY")
print("=" * 60)
for cls in CLASS_NAMES.values():
    sub = evaluatable[evaluatable["Ground_Truth"].str.contains(cls, na=False)]
    if len(sub) == 0:
        continue
    cor = len(sub[sub["Correct"] == "Correct"])
    print(f"  {cls:<20}: {cor:>4} / {len(sub):<4}  ({cor/len(sub)*100:.1f}%)")

print("\n" + "=" * 60)
print("🏷   PROXIMITY LEVEL DISTRIBUTION")
print("=" * 60)
print(primary["Proximity_Level"].value_counts().to_string())

print("\n" + "=" * 60)
print("📏  RESULTS BY DISTANCE LEVEL")
print("=" * 60)
for level in sorted(primary["Distance_Level"].unique()):
    sub = primary[primary["Distance_Level"] == level]
    ev  = sub[~sub["Correct"].isin(["No GT"])]
    cor = len(ev[ev["Correct"] == "Correct"])
    wrg = len(ev[ev["Correct"] == "Wrong"])
    acc = (cor / len(ev) * 100) if len(ev) else 0
    det = sub[sub["Predicted_Class"] != "No Detection"]
    print(f"\n  Level {level}  ({len(sub)} images):")

    print(f"    ✅ Correct   : {cor}")
    print(f"    ❌ Wrong     : {wrg}")
    print(f"    🎯 Accuracy  : {acc:.2f}%")
    if len(det):
        print(f"    Avg Conf    : {det['Confidence'].mean():.4f}")
        print(f"    Avg Occ%    : {det['Occupancy_%'].mean():.2f}%")

print("\n" + "=" * 60)
print("🔢  FINAL TALLY")
print("=" * 60)
print(f"  ✅  Total Correct  : {correct_n}")
print(f"  ❌  Total Wrong    : {wrong_n}")
print(f"  ❓  No GT          : {no_gt_n}")
print(f"  🎯  Final Accuracy : {accuracy:.2f}%")
print("=" * 60)

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Model input         : 640 × 640
Classes             : ['Speed_Breaker', 'Slippery_Road', 'School_Zone']
Confidence threshold: 0.25
Labels folder       : /content/drive/MyDrive/Major_Project/Test_4500/labels
Occupancy formula   : (box_w × box_h) / (img_w × img_h) × 100
Note                : Occupancy % + Background % = 100% always
Image                            Ground_Truth       Predicted_Class    Correct     Confidence  Occupancy%  Background%
❌ SB_l10_left_0001.jpg             Speed_Breaker      No Detection       Wrong           0.0000       0.00%      100.00%
❌ SB_l10_left_0002.jpg             Speed_Breaker      No Detection       Wrong           0.0000       0.00%      100.00%
❌ SB_l10_left_0003.jpg             Speed_Breaker      No Detection       Wrong           0.0000       0.00%      100.00%
❌ SB_l10_left_0004.jpg             Speed_Breaker      No Detection       Wrong           0.0000       0.00%      100.00%
❌ SB_l10_left_0005.jpg             Speed_Breaker      No Detectio

In [ ]:
import pandas as pd
from google.colab import drive



input_path  = "/content/drive/MyDrive/Major_Project/Test_4500/occupancy_results_mobilevit_SEblock/occupancy_results.csv"
output_path = "/content/drive/MyDrive/Major_Project/Test_4500/occupancy_results_mobilevit_SEblock/occupancy_summary.xlsx"

df = pd.read_csv(input_path)

# Convert to numeric
df["Occupancy_%"] = pd.to_numeric(df["Occupancy_%"], errors="coerce")
df["Confidence"]  = pd.to_numeric(df["Confidence"],  errors="coerce")

# ✅ Remove secondary detections (rows with "—")
df_primary = df[df["Correct"].isin(["Correct", "Wrong"])].copy()

bins   = [0, 3, 5, 10, 20, 30, 101]
labels = ["<3%", "3%", "5%", "10%", "20%", "30%"]

df_primary["Occ_Range"] = pd.cut(df_primary["Occupancy_%"], bins=bins, labels=labels, right=True)

# Count correct and wrong per bin
summary = df_primary.groupby(["Occ_Range", "Correct"], observed=True).size().unstack(fill_value=0).reset_index()
summary.columns.name = None

# Average confidence for correct only
avg_conf = df_primary[df_primary["Correct"] == "Correct"].groupby("Occ_Range", observed=True)["Confidence"].mean().reset_index()
avg_conf.columns = ["Occ_Range", "Avg_Confidence_%"]
avg_conf["Avg_Confidence_%"] = (avg_conf["Avg_Confidence_%"] * 100).round(2)

# Merge
result = summary.merge(avg_conf, on="Occ_Range")
result["Total"] = result.get("Correct", 0) + result.get("Wrong", 0)

print(result.to_string(index=False))

# ✅ Save to Excel
result.to_excel(output_path, index=False)
print(f"\n✅ Saved to: {output_path}")

Occ_Range  Correct  Wrong  Avg_Confidence_%  Total
      <3%      105     94             45.65    199
       3%       49    366             65.03    415
       5%      217    410             48.79    627
      10%       25    178             81.29    203
      20%       25     82             90.74    107
      30%       75     32             73.70    107

✅ Saved to: /content/drive/MyDrive/Major_Project/Test_4500/occupancy_results_mobilevit_SEblock/occupancy_summary.xlsx


In [ ]:
Occupancy code for train images alone

In [ ]:
# ============================================================
# Occupancy Analysis from YOLO Label Files (No Model Needed)
# Reads .txt label files directly to compute occupancy %
# Formula: (box_w × box_h) / (img_w × img_h) × 100
# ============================================================

import os
import re
import cv2
import numpy as np
import pandas as pd

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────

CLASS_NAMES = {
    0: "Speed_Breaker",
    1: "Slippery_Road",
    2: "School_Zone"
}

# Point these to your TRAIN folder
IMAGE_FOLDER   = "/content/drive/MyDrive/Major_Project/train/images"
LABELS_FOLDER  = "/content/drive/MyDrive/Major_Project/train/labels"
RESULTS_FOLDER = "/content/drive/MyDrive/Major_Project/train/occupancy_analysis"
ANNOTATED_FOLDER = os.path.join(RESULTS_FOLDER, "annotated_images")

os.makedirs(RESULTS_FOLDER, exist_ok=True)
os.makedirs(ANNOTATED_FOLDER, exist_ok=True)


# ─────────────────────────────────────────────
# OCCUPANCY FUNCTIONS
# ─────────────────────────────────────────────

def compute_occupancy(box_w_norm, box_h_norm):
    """
    YOLO labels store box width and height as fractions (0–1) of image size.
    So occupancy is simply:

        occupancy % = box_w_norm × box_h_norm × 100

    No need to multiply by image size because the normalization cancels out:
        (box_w_px × box_h_px) / (img_w × img_h) × 100
      = (box_w_norm × img_w × box_h_norm × img_h) / (img_w × img_h) × 100
      = box_w_norm × box_h_norm × 100

    This means occupancy can be computed directly from the label file
    without even opening the image.
    """
    occ = round(box_w_norm * box_h_norm * 100, 4)
    bg  = round(100.0 - occ, 4)
    return occ, bg


def occupancy_level(pct):
    if pct >= 50: return "VERY CLOSE"
    if pct >= 30: return "CLOSE"
    if pct >= 15: return "MODERATE"
    if pct >= 5:  return "FAR"
    return "VERY FAR"


def xywh_norm_to_xyxy_px(cx, cy, w, h, img_w, img_h):
    """Convert normalised YOLO box → pixel coordinates for drawing."""
    x1 = int((cx - w / 2) * img_w)
    y1 = int((cy - h / 2) * img_h)
    x2 = int((cx + w / 2) * img_w)
    y2 = int((cy + h / 2) * img_h)
    return (max(0, x1), max(0, y1),
            min(img_w - 1, x2), min(img_h - 1, y2))


# ─────────────────────────────────────────────
# ANNOTATION DRAWING
# ─────────────────────────────────────────────

def draw_annotations(image, annotations, img_name):
    """
    annotations: list of dicts with box, class_id, occupancy_pct, etc.
    """
    vis  = image.copy()
    h, w = vis.shape[:2]

    COLORS = {
        0: (0, 165, 255),   # Speed Breaker → orange
        1: (255, 0, 200),   # Slippery Road → pink
        2: (255, 220, 0),   # School Zone   → cyan
    }

    for ann in annotations:
        x1, y1, x2, y2 = ann["box_px"]
        color = COLORS.get(ann["class_id"], (200, 200, 200))

        # Bounding box (from label file — ground truth box)
        cv2.rectangle(vis, (x1, y1), (x2, y2), color, 2)

        # Label lines
        lines = [
            CLASS_NAMES[ann["class_id"]],
            f"Occ : {ann['occupancy_pct']:.4f}%",
            f"BG  : {ann['background_pct']:.4f}%",
            f"[{ann['level']}]",
        ]
        ty = max(y1 - 65, 14)
        for i, line in enumerate(lines):
            cv2.putText(vis, line, (x1, ty + i * 15),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.43, color, 2)

        # Semi-transparent fill
        overlay = vis.copy()
        cv2.rectangle(overlay, (x1, y1), (x2, y2), color, -1)
        cv2.addWeighted(overlay, 0.12, vis, 0.88, 0, vis)

    # Top banner — show best (largest) annotation
    if annotations:
        best   = max(annotations, key=lambda a: a["occupancy_pct"])
        banner = (f"GT: {CLASS_NAMES[best['class_id']]}  "
                  f"Occ={best['occupancy_pct']:.4f}%  "
                  f"BG={best['background_pct']:.4f}%  "
                  f"[{best['level']}]  "
                  f"| {len(annotations)} annotation(s)")
        bg_col = (30, 100, 30)
    else:
        banner = f"No label found for {img_name}"
        bg_col = (60, 60, 60)

    cv2.rectangle(vis, (0, 0), (w, 36), bg_col, -1)
    cv2.putText(vis, banner, (6, 24),
                cv2.FONT_HERSHEY_SIMPLEX, 0.48, (255, 255, 255), 2)

    # Bottom occupancy bar
    if annotations:
        best  = max(annotations, key=lambda a: a["occupancy_pct"])
        bar_y = h - 22
        occ_w = int(w * best["occupancy_pct"] / 100)
        cv2.rectangle(vis, (0, bar_y), (w, bar_y + 18), (40, 40, 40), -1)
        cv2.rectangle(vis, (0, bar_y), (occ_w, bar_y + 18), (0, 160, 80), -1)
        cv2.putText(vis,
                    f"Occupancy {best['occupancy_pct']:.4f}% | Background {best['background_pct']:.4f}%",
                    (6, bar_y + 13),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1)

    return vis


# ─────────────────────────────────────────────
# MAIN LOOP
# ─────────────────────────────────────────────

print(f"Images  folder : {IMAGE_FOLDER}")
print(f"Labels  folder : {LABELS_FOLDER}")
print(f"Results folder : {RESULTS_FOLDER}")
print(f"\nOccupancy formula : box_w_norm × box_h_norm × 100")
print(f"Note              : Occupancy % + Background % = 100% always")
print("=" * 100)
print(f"{'Image':<35} {'Class':<18} {'Occupancy %':>13} {'Background %':>14} {'Level':<14} {'Ann#':>5}")
print("=" * 100)

data             = []
total_images     = 0
images_with_label = 0
images_no_label  = 0

for img_name in sorted(os.listdir(IMAGE_FOLDER)):
    if not img_name.lower().endswith((".jpg", ".png", ".jpeg")):
        continue

    total_images += 1
    img_path   = os.path.join(IMAGE_FOLDER, img_name)
    base       = os.path.splitext(img_name)[0]
    label_path = os.path.join(LABELS_FOLDER, base + ".txt")

    # ── Read image (only needed for dimensions + annotated output) ──
    image = cv2.imread(img_path)
    if image is None:
        print(f"⚠  Could not read image: {img_name}")
        continue
    img_h, img_w = image.shape[:2]

    # ── Read label file ──────────────────────────────────────────────
    if not os.path.exists(label_path):
        print(f"❓ {img_name:<35} No label file found")
        images_no_label += 1

        data.append({
            "Image":          img_name,
            "Annotation_#":   0,
            "Class":          "No Label",
            "Class_ID":       -1,
            "YOLO_cx":        "",
            "YOLO_cy":        "",
            "YOLO_w":         "",
            "YOLO_h":         "",
            "Occupancy_%":    0.0,
            "Background_%":   100.0,
            "Occ+BG_Check":   100.0,
            "Proximity_Level": "NO LABEL",
            "Image_W":        img_w,
            "Image_H":        img_h,
            "Distance_Level": (lambda m: m.group(1) if m else "unknown")(
                               re.search(r'_l(\d+)_', img_name)),
        })

        vis = draw_annotations(image, [], img_name)
        cv2.imwrite(os.path.join(ANNOTATED_FOLDER, img_name), vis)
        continue

    images_with_label += 1

    # ── Parse each annotation line ───────────────────────────────────
    annotations = []
    with open(label_path, "r") as f:
        lines = [l.strip() for l in f if l.strip()]

    dist_match = re.search(r'_l(\d+)_', img_name)
    dist_level = dist_match.group(1) if dist_match else "unknown"

    for ann_idx, line in enumerate(lines):
        parts    = line.split()
        cls_id   = int(parts[0])
        cx, cy   = float(parts[1]), float(parts[2])
        bw, bh   = float(parts[3]), float(parts[4])   # normalised width, height

        occ_pct, bg_pct = compute_occupancy(bw, bh)
        level           = occupancy_level(occ_pct)
        box_px          = xywh_norm_to_xyxy_px(cx, cy, bw, bh, img_w, img_h)
        cls_name        = CLASS_NAMES.get(cls_id, f"Unknown_{cls_id}")

        annotations.append({
            "class_id":       cls_id,
            "box_px":         box_px,
            "occupancy_pct":  occ_pct,
            "background_pct": bg_pct,
            "level":          level,
        })

        # Print — first annotation on the image name line, rest indented
        prefix = f"  {img_name:<35}" if ann_idx == 0 else f"  {'':>35}"
        print(f"{prefix} {cls_name:<18} {occ_pct:>12.4f}% "
              f"{bg_pct:>13.4f}% {level:<14} {ann_idx + 1:>5}")

        data.append({
            "Image":           img_name,
            "Annotation_#":    ann_idx + 1,
            "Class":           cls_name,
            "Class_ID":        cls_id,
            "YOLO_cx":         round(cx, 6),
            "YOLO_cy":         round(cy, 6),
            "YOLO_w":          round(bw, 6),
            "YOLO_h":          round(bh, 6),
            "Occupancy_%":     occ_pct,
            "Background_%":    bg_pct,
            "Occ+BG_Check":    round(occ_pct + bg_pct, 4),
            "Proximity_Level": level,
            "Image_W":         img_w,
            "Image_H":         img_h,
            "Distance_Level":  dist_level,
        })

    # ── Save annotated image ─────────────────────────────────────────
    vis = draw_annotations(image, annotations, img_name)
    cv2.imwrite(os.path.join(ANNOTATED_FOLDER, img_name), vis)

print("=" * 100)

# ─────────────────────────────────────────────
# SAVE CSV
# ─────────────────────────────────────────────

df       = pd.DataFrame(data)
csv_path = os.path.join(RESULTS_FOLDER, "occupancy_from_labels.csv")

# Tally rows at the bottom
detected_df = df[df["Class"] != "No Label"]

sep = {col: "─" * 12 for col in df.columns}

tally_rows = [
    sep,
    {"Image": "TOTAL IMAGES",          "Annotation_#": total_images,         "Class": "", "Class_ID": "", "YOLO_cx": "", "YOLO_cy": "", "YOLO_w": "", "YOLO_h": "", "Occupancy_%": "", "Background_%": "", "Occ+BG_Check": "", "Proximity_Level": "", "Image_W": "", "Image_H": "", "Distance_Level": ""},
    {"Image": "IMAGES WITH LABELS",     "Annotation_#": images_with_label,    "Class": "", "Class_ID": "", "YOLO_cx": "", "YOLO_cy": "", "YOLO_w": "", "YOLO_h": "", "Occupancy_%": "", "Background_%": "", "Occ+BG_Check": "", "Proximity_Level": "", "Image_W": "", "Image_H": "", "Distance_Level": ""},
    {"Image": "IMAGES WITHOUT LABELS",  "Annotation_#": images_no_label,      "Class": "", "Class_ID": "", "YOLO_cx": "", "YOLO_cy": "", "YOLO_w": "", "YOLO_h": "", "Occupancy_%": "", "Background_%": "", "Occ+BG_Check": "", "Proximity_Level": "", "Image_W": "", "Image_H": "", "Distance_Level": ""},
    {"Image": "TOTAL ANNOTATIONS",      "Annotation_#": len(detected_df),     "Class": "", "Class_ID": "", "YOLO_cx": "", "YOLO_cy": "", "YOLO_w": "", "YOLO_h": "", "Occupancy_%": "", "Background_%": "", "Occ+BG_Check": "", "Proximity_Level": "", "Image_W": "", "Image_H": "", "Distance_Level": ""},
    {"Image": "MEAN OCCUPANCY %",       "Annotation_#": f"{detected_df['Occupancy_%'].mean():.4f}%", "Class": "", "Class_ID": "", "YOLO_cx": "", "YOLO_cy": "", "YOLO_w": "", "YOLO_h": "", "Occupancy_%": "", "Background_%": "", "Occ+BG_Check": "", "Proximity_Level": "", "Image_W": "", "Image_H": "", "Distance_Level": ""},
    {"Image": "MAX OCCUPANCY %",        "Annotation_#": f"{detected_df['Occupancy_%'].max():.4f}%",  "Class": "", "Class_ID": "", "YOLO_cx": "", "YOLO_cy": "", "YOLO_w": "", "YOLO_h": "", "Occupancy_%": "", "Background_%": "", "Occ+BG_Check": "", "Proximity_Level": "", "Image_W": "", "Image_H": "", "Distance_Level": ""},
    {"Image": "MIN OCCUPANCY %",        "Annotation_#": f"{detected_df['Occupancy_%'].min():.4f}%",  "Class": "", "Class_ID": "", "YOLO_cx": "", "YOLO_cy": "", "YOLO_w": "", "YOLO_h": "", "Occupancy_%": "", "Background_%": "", "Occ+BG_Check": "", "Proximity_Level": "", "Image_W": "", "Image_H": "", "Distance_Level": ""},
]

df_final = pd.concat([df, pd.DataFrame(tally_rows)], ignore_index=True)
df_final.to_csv(csv_path, index=False)

# ─────────────────────────────────────────────
# SUMMARY REPORT
# ─────────────────────────────────────────────

print(f"\n✅  Done!")
print(f"📂  CSV       : {csv_path}")
print(f"🖼   Annotated : {ANNOTATED_FOLDER}\n")

print("=" * 60)
print("📊  DATASET OCCUPANCY SUMMARY")
print("=" * 60)
print(f"  Total Images          : {total_images}")
print(f"  Images with labels    : {images_with_label}")
print(f"  Images without labels : {images_no_label}")
print(f"  Total annotations     : {len(detected_df)}")

print("\n" + "=" * 60)
print("📐  OCCUPANCY % STATISTICS")
print("=" * 60)
print(f"  Note: Occupancy % + Background % = 100% always\n")
print(f"  Mean   : {detected_df['Occupancy_%'].mean():.4f}%")
print(f"  Median : {detected_df['Occupancy_%'].median():.4f}%")
print(f"  Min    : {detected_df['Occupancy_%'].min():.4f}%")
print(f"  Max    : {detected_df['Occupancy_%'].max():.4f}%")
print(f"  Std    : {detected_df['Occupancy_%'].std():.4f}%")

print("\n" + "=" * 60)
print("🏷   CLASS-WISE OCCUPANCY")
print("=" * 60)
for cls in CLASS_NAMES.values():
    sub = detected_df[detected_df["Class"] == cls]
    if len(sub) == 0:
        continue
    print(f"\n  {cls}  ({len(sub)} annotations):")
    print(f"    Mean Occ : {sub['Occupancy_%'].mean():.4f}%")
    print(f"    Min  Occ : {sub['Occupancy_%'].min():.4f}%")
    print(f"    Max  Occ : {sub['Occupancy_%'].max():.4f}%")

print("\n" + "=" * 60)
print("🏷   PROXIMITY LEVEL DISTRIBUTION")
print("=" * 60)
print(detected_df["Proximity_Level"].value_counts().to_string())

print("\n" + "=" * 60)
print("📏  OCCUPANCY BY DISTANCE LEVEL")
print("=" * 60)
for level in sorted(detected_df["Distance_Level"].unique()):
    sub = detected_df[detected_df["Distance_Level"] == level]
    print(f"\n  Level {level}  ({len(sub)} annotations):")
    print(f"    Mean Occ : {sub['Occupancy_%'].mean():.4f}%")
    print(f"    Min  Occ : {sub['Occupancy_%'].min():.4f}%")
    print(f"    Max  Occ : {sub['Occupancy_%'].max():.4f}%")
    for lvl, cnt in sub["Proximity_Level"].value_counts().items():
        print(f"    {lvl:<14}: {cnt} images")

print("\n" + "=" * 60)
print("🔍  PER-IMAGE OCCUPANCY")
print("=" * 60)
print(f"  {'Image':<35} {'Class':<18} {'Occ%':>10} {'BG%':>10} {'Level'}")
print(f"  {'-'*35} {'-'*18} {'-'*10} {'-'*10} {'-'*14}")
for _, row in df[df["Class"] != "No Label"].iterrows():
    print(f"  {str(row['Image']):<35} {str(row['Class']):<18} "
          f"{float(row['Occupancy_%']):>9.4f}% "
          f"{float(row['Background_%']):>9.4f}% "
          f"{row['Proximity_Level']}")

print("=" * 60)

In [ ]:
import pandas as pd

input_path  = "/content/drive/MyDrive/Major_Project/Dataset_60_20_20_split/occupancy_analysis_Train/occupancy_from_labels.csv"
output_path = "/content/drive/MyDrive/Major_Project/Dataset_60_20_20_split/occupancy_analysis_Train/occupancy_summary.xlsx"

df = pd.read_csv(input_path)

# Convert to numeric, drop rows that can't be converted (tally/separator rows)
df["Occupancy_%"] = pd.to_numeric(df["Occupancy_%"], errors="coerce")
df = df[df["Occupancy_%"].notna()].copy()

print(f"Columns in your CSV : {list(df.columns)}")
print(f"Total rows loaded   : {len(df)}\n")

bins   = [0, 3, 5, 10, 20, 30, 101]
labels = ["<3%", "3%", "5%", "10%", "20%", "30%"]

# ── CASE 1: CSV has Correct/Wrong column (from model prediction code) ──────
if "Correct" in df.columns:
    df_primary = df[df["Correct"].isin(["Correct", "Wrong"])].copy()
    df_primary["Occ_Range"] = pd.cut(df_primary["Occupancy_%"],
                                      bins=bins, labels=labels, right=True)

    summary = (df_primary
               .groupby(["Occ_Range", "Correct"], observed=True)
               .size()
               .unstack(fill_value=0)
               .reset_index())
    summary.columns.name = None

    if "Correct" not in summary.columns:
        summary["Correct"] = 0
    if "Wrong" not in summary.columns:
        summary["Wrong"] = 0

    summary["Total"] = summary["Correct"] + summary["Wrong"]
    summary["%"]     = ((summary["Correct"] / summary["Total"]) * 100).round(2)

    result = summary[["Occ_Range", "Correct", "Wrong", "%", "Total"]]

# ── CASE 2: CSV has no Correct column (from labels-only occupancy code) ────
else:
    print("ℹ️  No 'Correct' column found — building occupancy-only summary\n")

    # Drop rows with no detection
    df_valid = df[df["Occupancy_%"] > 0].copy()
    df_valid["Occ_Range"] = pd.cut(df_valid["Occupancy_%"],
                                    bins=bins, labels=labels, right=True)

    summary = (df_valid
               .groupby("Occ_Range", observed=True)
               .agg(
                   Total      = ("Occupancy_%", "count"),
                   Avg_Occ    = ("Occupancy_%", "mean"),
                   Min_Occ    = ("Occupancy_%", "min"),
                   Max_Occ    = ("Occupancy_%", "max"),
               )
               .reset_index())

    summary["Avg_Occ"] = summary["Avg_Occ"].round(4)
    summary["Min_Occ"] = summary["Min_Occ"].round(4)
    summary["Max_Occ"] = summary["Max_Occ"].round(4)
    summary["%_of_Total"] = ((summary["Total"] / summary["Total"].sum()) * 100).round(2)

    result = summary[["Occ_Range", "Total", "%_of_Total", "Avg_Occ", "Min_Occ", "Max_Occ"]]

print(result.to_string(index=False))

# Save to Excel
result.to_excel(output_path, index=False)
print(f"\n✅ Saved to: {output_path}")

Columns in your CSV : ['Image', 'Annotation_#', 'Class', 'Class_ID', 'YOLO_cx', 'YOLO_cy', 'YOLO_w', 'YOLO_h', 'Occupancy_%', 'Background_%', 'Occ+BG_Check', 'Proximity_Level', 'Image_W', 'Image_H', 'Distance_Level']
Total rows loaded   : 1907

ℹ️  No 'Correct' column found — building occupancy-only summary

Occ_Range  Total  %_of_Total  Avg_Occ  Min_Occ  Max_Occ
      <3%    101        5.30   0.9954   0.3910   2.0597
       3%     18        0.94   4.7047   4.2654   4.9820
       5%     60        3.15   7.1058   5.0230   9.7538
      10%    133        6.97  13.6404  10.0295  19.8511
      20%     38        1.99  24.5149  20.2298  28.2102
      30%   1557       81.65  62.4458  47.2846  90.4659

✅ Saved to: /content/drive/MyDrive/Major_Project/Dataset_60_20_20_split/occupancy_analysis_Train/occupancy_summary.xlsx
